# UdaPlay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [ ]:
# Optional SQLite compatibility for the Udacity workspace.
# pysqlite3 is imported dynamically so local editors do not warn when it is absent.

from importlib import import_module, util
import sys

if util.find_spec("pysqlite3") is not None:
    pysqlite3 = import_module("pysqlite3")
    sys.modules["sqlite3"] = pysqlite3

In [ ]:
import os

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.agents import Agent
from lib.evaluation import AgentEvaluator, TestCase
from lib.llm import LLM
from lib.messages import AIMessage, SystemMessage, ToolMessage, UserMessage
from lib.parsers import PydanticOutputParser
from lib.tooling import tool
from lib.vector_db import VectorStore


In [ ]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openai.vocareum.com/v1")

assert OPENAI_API_KEY is not None, "OPENAI_API_KEY must be set in your local .env file"
assert TAVILY_API_KEY is not None, "TAVILY_API_KEY must be set in your local .env file"

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [ ]:
from itertools import zip_longest

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    api_base=OPENAI_BASE_URL,
)

chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)
game_vector_store = VectorStore(collection)


@tool
def retrieve_game(query: str, n_results: int = 3) -> list[dict]:
    """Semantic search: Finds most similar game records in the vector DB.

    args:
    - query: a question about the game industry.
    - n_results: number of local game records to return.
    """
    results = game_vector_store.query(query_texts=[query], n_results=n_results)
    documents = results.get("documents", [[]])[0] or []
    metadatas = results.get("metadatas", [[]])[0] or []
    distances = results.get("distances", [[]])[0] or []
    ids = results.get("ids", [[]])[0] or []

    games = []
    for document, metadata, distance, game_id in zip_longest(
        documents,
        metadatas,
        distances,
        ids,
        fillvalue=None,
    ):
        metadata = metadata or {}
        games.append({
            "id": game_id,
            "Platform": metadata.get("Platform"),
            "Name": metadata.get("Name"),
            "YearOfRelease": metadata.get("YearOfRelease"),
            "Genre": metadata.get("Genre"),
            "Publisher": metadata.get("Publisher"),
            "Description": metadata.get("Description", document),
            "document": document,
            "distance": distance,
        })

    return games


#### Evaluate Retrieval Tool

In [ ]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

#### Game Web Search Tool

In [ ]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

### Agent

In [ ]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

In [ ]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes